# 含持有期零段反转的八列表

本 Notebook 先调用当前最终冻结包生成原始八列表，再只替换 `0转-1` 和 `0转+1` 两列。

- 零段信号在形成日收盘计算，下一实际交易日执行。
- 执行日的零段信号使用当前冻结参数中的完整持仓机路径，信号发出后连续输出 1。下侧使用 H03，上侧使用 H04；最短持有、最长持有、释放阈值和冷却期均沿用冻结参数。
- 如果持有路径对应的基础三状态不是 0，则把该日零段信号置为 0。
- `0转-1` 与 `0转+1` 同时为 1 的情况暂不处理，保留两列各自的冻结持有结果。
- `三状态`、`+1反转`、`-1反转`、`大涨`、`大跌` 不做修改。

运行前请确保环境使用冻结包要求的 `pandas>=2.0,<3.0`，并设置 `COMPANY_SPOT_PATH`。

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'src' / 'generate_compact_output.py').is_file()
)
SRC_ROOT = PACKAGE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

SPOT_TEXT = os.environ.get('COMPANY_SPOT_PATH', '').strip()
if not SPOT_TEXT:
    raise RuntimeError(
        '请先设置 COMPANY_SPOT_PATH 为唯一的原始现货 Parquet/CSV/TSV 文件路径。'
    )

from generate_compact_output import generate_compact_output, resolve_spot

SPOT_PATH = resolve_spot(SPOT_TEXT)
RUN_ROOT = Path(
    os.environ.get(
        'HOLDING_OUTPUT_DIR',
        str(PACKAGE_ROOT / 'runtime_outputs_holding_period'),
    )
).expanduser().resolve()
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print(f'冻结包目录：{PACKAGE_ROOT}')
print(f'现货输入：{SPOT_PATH}')
print(f'输出目录：{RUN_ROOT}')

冻结包目录：C:\Users\admin\Desktop\三状态增强\最终冻结运行上传包_含零段反转_20260820_1350
现货输入：C:\Users\admin\Desktop\三状态增强\05_上传包_日期解析修正版\data\local_smoke\CSI500_SPOT_000905_XSHG_20070115_20260817_audited_pure_spot_fixture.parquet
输出目录：C:\Users\admin\Desktop\三状态增强\_tmp_hold_test\output_rerun


## 1. 生成当前冻结包的原始八列表

这一格调用生产入口，生成未加入持有期的原始八列表。它不会重选候选参数。

In [2]:
record = generate_compact_output(SPOT_PATH, RUN_ROOT)
EVENT_EIGHT_PATH = RUN_ROOT / '最终执行日简表.csv'
event_eight = pd.read_csv(EVENT_EIGHT_PATH, encoding='utf-8-sig')
event_eight['实际执行日'] = pd.to_datetime(event_eight['实际执行日'], errors='raise').dt.normalize()

COMPACT_COLUMNS = ['实际执行日', '三状态', '+1反转', '-1反转', '0转-1', '0转+1', '大涨', '大跌']
if list(event_eight.columns) != COMPACT_COLUMNS:
    raise AssertionError(f'原始八列表列顺序不符：{list(event_eight.columns)}')

print(f'原始八列表：{EVENT_EIGHT_PATH}')
print(f'行数：{len(event_eight):,}')
display(event_eight.head())

[非零冻结] 启动


[非零退出] 读取冻结 V55；只计算最终 score_02，不重建候选池
[非零退出] V55 冻结信号完成；事件数=45
[非零退出] 读取冻结 V80；只计算最终 score_02，不重建候选池


[非零退出] V80 冻结信号完成；事件数=62
[非零退出] output=C:\Users\admin\Desktop\三状态增强\_tmp_hold_test\output_rerun\_engine_outputs\remote_nonzero_five_columns.csv
[非零退出] rows=2092 date=2018-01-03 -> 2026-08-18
[零段反转冻结] 启动


[零段反转] 读取唯一远端现货并从冻结八状态公式生成基础状态


[零段反转-down] 运行冻结 V38_down_s01_a1_q0.85_c1_H03（不扫描候选、不读取未来标签）
[零段反转-up] 运行冻结 V57_up_s01_a1_q0.85_c1_H04（不扫描候选、不读取未来标签）


...

[O2O up 17:30:14] 1/4 读取唯一现货并构造因果特征


[O2O up 17:30:16] 1/4 完成：rows=4761，Development=1213，Validation=482
[O2O up 17:30:16] 2/4 只计算已冻结参数：V189:base_1839_cov_0.055；不重建候选池


[O2O up 17:30:17] 3/4 用固定阈值生成冻结预测，之后才读取 Test
[O2O up 17:30:17] 4/4 冻结参数逐日结果已生成
[大涨大跌-up] prediction_output=C:\Users\admin\Desktop\三状态增强\_tmp_hold_test\output_rerun\_engine_outputs\remote_up_extreme_predictions.csv
[大涨大跌-up] formation_rows=2092 formation_date=2018-01-02 -> 2026-08-17


原始八列表：C:\Users\admin\Desktop\三状态增强\_tmp_hold_test\output_rerun\最终执行日简表.csv
行数：2,092


,实际执行日,三状态,+1反转,-1反转,0转-1,0转+1,大涨,大跌
0,2018-01-03,0,0,0,0,0,0,0
1,2018-01-04,0,0,0,0,0,0,0
2,2018-01-05,0,0,0,0,0,0,0
3,2018-01-08,0,0,0,0,0,0,0
4,2018-01-09,0,0,0,0,0,0,0


## 2. 重建冻结零段信号的连续持有路径

这里直接调用冻结运行包中的评分和持仓机函数，不读取未来收益标签。`holding_path` 是冻结参数实际产生的连续持有路径。

In [3]:
from run_zero_transfer_frozen import (
    FROZEN_ZERO_TRANSFER,
    _effective_dates,
    _frozen_signal_path,
)
from spot_panel import load_spot_panel
from zero_transfer.logic_features import compute_logic_scores

spot, panel, spot_audit = load_spot_panel(SPOT_PATH)
effective_dates, pending_effective = _effective_dates(panel)

entry_paths = {}
holding_paths = {}
freeze_rows = {}
for side in ('down', 'up'):
    freeze = FROZEN_ZERO_TRANSFER[side]
    scores = compute_logic_scores(
        str(freeze['source_version']),
        spot,
        panel,
        int(freeze['direction']),
    )
    score_column = str(freeze['score_column'])
    if score_column not in scores.columns:
        raise KeyError(f'{side} 冻结评分列不存在：{score_column}')
    selected, holding = _frozen_signal_path(
        panel,
        scores[score_column].to_numpy(dtype=float),
        freeze,
    )
    entry_paths[side] = selected.astype(bool)
    holding_paths[side] = holding.astype(bool)
    freeze_rows[side] = freeze

base_state = pd.to_numeric(panel['state'], errors='raise').astype(int).to_numpy()
base_is_zero = base_state == 0

# 先保留冻结持有路径，再按执行日对应的基础状态做冲突置零。
down_conflict = holding_paths['down'] & ~base_is_zero
up_conflict = holding_paths['up'] & ~base_is_zero
down_holding = holding_paths['down'] & base_is_zero
up_holding = holding_paths['up'] & base_is_zero

holding_detail = pd.DataFrame({
    '形成日': pd.to_datetime(panel['formation_date'], errors='raise').dt.normalize(),
    '实际执行日': pd.to_datetime(effective_dates),
    '基础三状态': base_state,
    '下侧事件信号': entry_paths['down'].astype('int8'),
    '下侧冻结持有路径': holding_paths['down'].astype('int8'),
    '0转-1最终信号': down_holding.astype('int8'),
    '上侧事件信号': entry_paths['up'].astype('int8'),
    '上侧冻结持有路径': holding_paths['up'].astype('int8'),
    '0转+1最终信号': up_holding.astype('int8'),
})

if holding_detail['实际执行日'].duplicated().any():
    raise AssertionError('持有期明细的实际执行日重复')

display(pd.DataFrame([
    {
        '方向': side,
        '候选编号': str(freeze_rows[side]['candidate_id']),
        '持有包': str(freeze_rows[side]['holding_package']['package_id']),
        '最短持有日': int(freeze_rows[side]['holding_package']['min_hold_days']),
        '最长持有日': int(freeze_rows[side]['holding_package']['max_hold_days']),
        '事件信号数': int(entry_paths[side].sum()),
        '冻结持有日数': int(holding_paths[side].sum()),
        '基础状态冲突后置零日数': int((down_conflict if side == 'down' else up_conflict).sum()),
    }
    for side in ('down', 'up')
]))

,方向,候选编号,持有包,最短持有日,最长持有日,事件信号数,冻结持有日数,基础状态冲突后置零日数
0,down,V38_down_s01_a1_q0.85_c1_H03,H03,3,10,136,331,9
1,up,V57_up_s01_a1_q0.85_c1_H04,H04,5,20,86,420,30


## 3. 合并为含持有期的八列表

只替换两个零段列；两个零段方向同日同时为 1 时不做额外冲突处理。

In [4]:
detail_by_execution = holding_detail.set_index('实际执行日')
event_dates = pd.DatetimeIndex(event_eight['实际执行日'])
if set(event_dates) != set(detail_by_execution.index):
    raise AssertionError('原始八列表与持有期明细的执行日集合不一致')

# 官方事件信号应当与同一冻结路径的 entry_path 一致；先做日期对齐审计。
entry_check = detail_by_execution.reindex(event_dates)
if not np.array_equal(event_eight['三状态'].to_numpy(dtype=int), entry_check['基础三状态'].to_numpy(dtype=int)):
    raise AssertionError('基础三状态的形成日值与八列表执行日映射不一致')
if not np.array_equal(event_eight['0转-1'].to_numpy(dtype=int), entry_check['下侧事件信号'].to_numpy(dtype=int)):
    raise AssertionError('0转-1 事件信号与冻结运行结果不一致')
if not np.array_equal(event_eight['0转+1'].to_numpy(dtype=int), entry_check['上侧事件信号'].to_numpy(dtype=int)):
    raise AssertionError('0转+1 事件信号与冻结运行结果不一致')

holding_eight = event_eight.copy()
holding_eight['0转-1'] = entry_check['0转-1最终信号'].to_numpy(dtype='int8')
holding_eight['0转+1'] = entry_check['0转+1最终信号'].to_numpy(dtype='int8')

for column in ('三状态', '+1反转', '-1反转', '大涨', '大跌'):
    if not np.array_equal(holding_eight[column].to_numpy(), event_eight[column].to_numpy()):
        raise AssertionError(f'{column} 不应被持有期口径修改')

OUTPUT_PATH = RUN_ROOT / '含持有期八列表.csv'
DETAIL_PATH = RUN_ROOT / '零段反转持有期明细.csv'
METADATA_PATH = RUN_ROOT / '含持有期八列表_运行记录.json'

holding_eight.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
holding_detail.to_csv(DETAIL_PATH, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')

both_entry_or_holding = (holding_eight['0转-1'].eq(1) & holding_eight['0转+1'].eq(1))
metadata = {
    'input_spot': str(SPOT_PATH),
    'source_event_eight': str(EVENT_EIGHT_PATH),
    'output_file': str(OUTPUT_PATH),
    'holding_detail_file': str(DETAIL_PATH),
    'columns': COMPACT_COLUMNS,
    'date_rule': 'formation_date t close -> actual execution date t+1; hold path starts on the execution date',
    'zero_transfer_policy': 'use frozen holding_path, then set the zero-transfer value to 0 when the aligned base three-state is not 0',
    'same_day_down_up_policy': 'not handled; retain both independently generated columns',
    'same_day_both_one_days': int(both_entry_or_holding.sum()),
    'pending_latest_execution_display': bool(pending_effective),
    'freeze': {
        side: {
            'candidate_id': str(freeze_rows[side]['candidate_id']),
            'source_version': str(freeze_rows[side]['source_version']),
            'direction': int(freeze_rows[side]['direction']),
            'holding_package': dict(freeze_rows[side]['holding_package']),
            'threshold_from_development': float(freeze_rows[side]['threshold_from_development']),
            'release_threshold': float(freeze_rows[side]['release_threshold']),
        }
        for side in ('down', 'up')
    },
    'counts': {
        'rows': int(len(holding_eight)),
        'down_event_days': int(entry_paths['down'].sum()),
        'up_event_days': int(entry_paths['up'].sum()),
        'down_raw_holding_days': int(holding_paths['down'].sum()),
        'up_raw_holding_days': int(holding_paths['up'].sum()),
        'down_conflict_zeroed_days': int(down_conflict.sum()),
        'up_conflict_zeroed_days': int(up_conflict.sum()),
        'down_final_one_days': int(holding_eight['0转-1'].sum()),
        'up_final_one_days': int(holding_eight['0转+1'].sum()),
    },
}
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print(f'已输出：{OUTPUT_PATH}')
print(f'持有期明细：{DETAIL_PATH}')
print(f'运行记录：{METADATA_PATH}')
display(holding_eight.tail(20))

已输出：C:\Users\admin\Desktop\三状态增强\_tmp_hold_test\output_rerun\含持有期八列表.csv
持有期明细：C:\Users\admin\Desktop\三状态增强\_tmp_hold_test\output_rerun\零段反转持有期明细.csv
运行记录：C:\Users\admin\Desktop\三状态增强\_tmp_hold_test\output_rerun\含持有期八列表_运行记录.json


,实际执行日,三状态,+1反转,-1反转,0转-1,0转+1,大涨,大跌
2072,2026-07-22,-1,0,1,0,0,0,0
2073,2026-07-23,0,0,0,0,0,0,0
2074,2026-07-24,0,0,0,0,0,0,0
2075,2026-07-27,-1,0,0,0,0,0,0
2076,2026-07-28,-1,0,0,0,0,0,0
2077,2026-07-29,-1,0,0,0,0,0,0
2078,2026-07-30,-1,0,0,0,0,0,0
2079,2026-07-31,-1,0,0,0,0,0,0
2080,2026-08-03,-1,0,1,0,0,0,0
2081,2026-08-04,0,0,0,0,0,0,0


## 4. 最终检查

In [5]:
if list(holding_eight.columns) != COMPACT_COLUMNS:
    raise AssertionError('最终八列表列顺序不正确')
if holding_eight['实际执行日'].duplicated().any():
    raise AssertionError('最终八列表执行日重复')
if not holding_eight['实际执行日'].is_monotonic_increasing:
    raise AssertionError('最终八列表执行日未排序')
if not holding_eight['三状态'].isin([-1, 0, 1]).all():
    raise AssertionError('三状态出现非法值')
if not holding_eight[['+1反转', '-1反转', '0转-1', '0转+1', '大涨', '大跌']].isin([0, 1]).all().all():
    raise AssertionError('八列表信号出现非 0/1 值')
if ((holding_eight['三状态'].ne(0)) & holding_eight['0转-1'].eq(1)).any():
    raise AssertionError('0转-1 仍存在基础状态冲突')
if ((holding_eight['三状态'].ne(0)) & holding_eight['0转+1'].eq(1)).any():
    raise AssertionError('0转+1 仍存在基础状态冲突')

print('全部检查通过。')
print('注意：0转-1 与 0转+1 同时为 1 的情况按要求暂未处理。')
display(pd.DataFrame({
    '列': ['0转-1', '0转+1'],
    '事件日 1 数量': [int(event_eight['0转-1'].sum()), int(event_eight['0转+1'].sum())],
    '持有期最终 1 数量': [int(holding_eight['0转-1'].sum()), int(holding_eight['0转+1'].sum())],
    '基础状态冲突置零': [int(down_conflict.sum()), int(up_conflict.sum())],
}))

全部检查通过。
注意：0转-1 与 0转+1 同时为 1 的情况按要求暂未处理。


,列,事件日 1 数量,持有期最终 1 数量,基础状态冲突置零
0,0转-1,136,322,9
1,0转+1,86,390,30
